# Ghi nhớ bài học: Evaluation & Observability cho LLM Apps với Opik

Notebook này tóm tắt bài học **“A Practical Guide to Integrate Evaluation and Observability into LLM Apps”** theo kiểu dễ nhớ, có ví dụ code nhỏ chạy offline.

> Mục tiêu: sau notebook này, bạn có thể giải thích được vì sao LLM app cần evaluation + observability, Opik giúp theo dõi những gì, và một workflow đánh giá RAG cơ bản gồm những bước nào.

## Cách học gợi ý

1. Đọc checklist ở mỗi phần.
2. Chạy các cell code từ trên xuống dưới.
3. Sửa dữ liệu mẫu để tự mô phỏng lỗi hallucination, retrieval kém, hoặc câu trả lời không liên quan.


## 1. Ý chính cần nhớ

Khi LLM app đi vào thực tế, ta không chỉ cần model trả lời được, mà cần biết:

- Câu trả lời có **đúng** không?
- Có **liên quan** đến câu hỏi không?
- Có **dựa trên context** hay tự bịa không?
- Pipeline RAG lấy đúng tài liệu chưa?
- Lệnh gọi LLM tốn bao nhiêu token, chi phí, thời gian?
- Khi app fail, fail ở bước retrieval, prompt, model generation, hay evaluation?

Hai khái niệm trung tâm:

| Khái niệm | Vai trò | Câu hỏi trả lời |
|---|---|---|
| Evaluation | Đo chất lượng output | “App trả lời tốt đến đâu?” |
| Observability | Nhìn vào bên trong pipeline | “Vì sao app trả lời như vậy?” |

**Opik** là framework open-source giúp trace, log, evaluate và quan sát LLM workflows như function call, OpenAI/Ollama call, LlamaIndex RAG pipeline, experiment và dashboard.


In [ ]:
from pprint import pprint
from statistics import mean


## 2. Mental model: một LLM app cần trace những gì?

Một request trong LLM app thường có nhiều bước:

1. User gửi câu hỏi.
2. App có thể rewrite query hoặc route request.
3. Nếu là RAG, app retrieve context từ vector database.
4. App tạo prompt cuối cùng.
5. LLM sinh câu trả lời.
6. Hệ thống log token, latency, output, context và score.

Observability tốt nghĩa là ta có thể mở từng request và xem lại toàn bộ chuỗi này.


In [ ]:
trace_steps = [
    {"step": "input", "what_to_log": "user question", "why": "biết request gốc là gì"},
    {"step": "retrieval", "what_to_log": "retrieved chunks + scores", "why": "debug RAG lấy đúng/sai context"},
    {"step": "prompt", "what_to_log": "final prompt", "why": "kiểm tra instruction và context đưa vào model"},
    {"step": "generation", "what_to_log": "model, output, tokens, latency", "why": "theo dõi chất lượng, chi phí, tốc độ"},
    {"step": "evaluation", "what_to_log": "metric scores + reasons", "why": "biết output tốt/xấu theo tiêu chí nào"},
]
pprint(trace_steps)


## 3. Opik trong bài học làm được gì?

Các khả năng chính được minh họa:

- Dùng `@track` để trace một Python function bình thường.
- Dùng `track_openai` để log OpenAI-compatible calls, bao gồm cả OpenAI và Ollama local server.
- Kết nối với LlamaIndex qua callback handler để trace RAG pipeline.
- Tạo evaluation dataset gồm `question`, `expected_answer`, `expected_context`.
- Chạy experiment trên dataset.
- Dùng metrics như relevance, hallucination, correctness/factuality, context-related scores.
- Xem dashboard để debug từng sample và từng metric.


In [ ]:
# Mini decorator mô phỏng ý tưởng @track của Opik, chạy offline để dễ học.
from functools import wraps
from time import perf_counter

TRACE_LOG = []

def track(name=None):
    def decorator(fn):
        @wraps(fn)
        def wrapper(*args, **kwargs):
            start = perf_counter()
            output = fn(*args, **kwargs)
            TRACE_LOG.append({
                "name": name or fn.__name__,
                "inputs": {"args": args, "kwargs": kwargs},
                "output": output,
                "latency_ms": round((perf_counter() - start) * 1000, 3),
            })
            return output
        return wrapper
    return decorator

@track("add_two_numbers")
def add(a, b):
    return a + b

add(2, 3)
add(10, 7)
pprint(TRACE_LOG)


## 4. Workflow evaluation cho RAG

Một evaluation workflow tối thiểu gồm:

1. Chuẩn bị dataset test.
2. Chạy app trên từng câu hỏi.
3. So sánh output với expected answer.
4. So sánh retrieved context với expected context.
5. Ghi score, reason và trace để debug.

Trong production, dataset này có thể được bổ sung từ các case thật mà app trả lời sai hoặc gây nghi ngờ.


In [ ]:
eval_dataset = [
    {
        "question": "Opik dùng để làm gì trong LLM app?",
        "expected_answer": "Opik giúp trace, evaluate và observe LLM applications.",
        "expected_context": "Opik is an open-source LLM evaluation and observability framework."
    },
    {
        "question": "Vì sao RAG cần log retrieved context?",
        "expected_answer": "Để debug xem hệ thống có lấy đúng thông tin trước khi sinh câu trả lời hay không.",
        "expected_context": "RAG evaluation should compare expected context with retrieved context."
    },
    {
        "question": "Evaluation khác observability thế nào?",
        "expected_answer": "Evaluation đo chất lượng output, còn observability giúp nhìn vào bên trong pipeline để tìm nguyên nhân.",
        "expected_context": "Evaluation ensures quality; observability captures inner workings and bottlenecks."
    },
]
pprint(eval_dataset)


## 5. Mini RAG app giả lập

Cell dưới đây không gọi LLM thật. Nó mô phỏng một RAG app để ta tập nhớ workflow:

- `retrieve_context(question)` chọn context gần đúng bằng keyword đơn giản.
- `my_llm_application(question)` sinh câu trả lời template.
- Decorator `@track` lưu lại input, output và latency.

Trong app thật, phần này có thể là LlamaIndex + OpenAI/Ollama và được Opik trace tự động.


In [ ]:
knowledge_base = [
    "Opik is an open-source LLM evaluation and observability framework.",
    "RAG evaluation should compare expected context with retrieved context.",
    "Evaluation ensures quality; observability captures inner workings and bottlenecks.",
]

def retrieve_context(question):
    q = question.lower()
    if "rag" in q or "context" in q:
        return knowledge_base[1]
    if "evaluation" in q or "observability" in q or "khác" in q:
        return knowledge_base[2]
    return knowledge_base[0]

@track("mini_rag_app")
def my_llm_application(question):
    context = retrieve_context(question)
    if "opik" in question.lower():
        answer = "Opik giúp trace, evaluate và observe LLM applications."
    elif "rag" in question.lower() or "context" in question.lower():
        answer = "RAG cần log retrieved context để debug retrieval và kiểm tra nguồn thông tin."
    else:
        answer = "Evaluation đo chất lượng output; observability giúp debug các bước bên trong pipeline."
    return {"answer": answer, "retrieved_context": context}

results = []
for row in eval_dataset:
    pred = my_llm_application(row["question"])
    results.append({**row, **pred})

pprint(results)


## 6. Metrics đơn giản để nhớ ý tưởng

Bài học dùng các metric LLM-evaluator phức tạp hơn. Ở đây ta dùng metric keyword overlap để minh họa:

- `answer_overlap`: output có giống expected answer không?
- `context_match`: retrieved context có trùng expected context không?
- `hallucination_risk`: rủi ro hallucination cao nếu answer dùng ít từ xuất hiện trong context.

Metric thật trong Opik có thể dùng LLM-as-a-judge và trả về cả score lẫn reason.


In [ ]:
def tokenize(text):
    punctuation = ".,;:!?()[]{}\"'"
    tokens = set()
    for word in text.split():
        cleaned = word.strip(punctuation).lower()
        if len(cleaned) > 2:
            tokens.add(cleaned)
    return tokens

def overlap_score(a, b):
    a_tokens, b_tokens = tokenize(a), tokenize(b)
    if not a_tokens or not b_tokens:
        return 0.0
    return len(a_tokens & b_tokens) / len(a_tokens | b_tokens)

eval_results = []
for r in results:
    row = dict(r)
    row["answer_overlap"] = overlap_score(row["answer"], row["expected_answer"])
    row["context_match"] = float(row["retrieved_context"] == row["expected_context"])
    row["hallucination_risk"] = 1 - overlap_score(row["answer"], row["retrieved_context"])
    eval_results.append(row)

pprint([{k: r[k] for k in ["question", "answer_overlap", "context_match", "hallucination_risk"]} for r in eval_results])


In [ ]:
summary = {
    "answer_overlap": mean(r["answer_overlap"] for r in eval_results),
    "context_match": mean(r["context_match"] for r in eval_results),
    "hallucination_risk": mean(r["hallucination_risk"] for r in eval_results),
}
pprint(summary)


## 7. Debug như trên Opik dashboard

Khi một sample có score xấu, ta cần mở trace để xem:

- Input question là gì?
- Retrieved context có đúng không?
- Final answer có bám vào context không?
- Metric nào thấp?
- Reason của metric là gì?

Cell dưới đây lọc ra sample có rủi ro hallucination cao nhất để tập debug.


In [ ]:
worst = max(eval_results, key=lambda r: r["hallucination_risk"])
print("QUESTION:", worst["question"])
print("EXPECTED CONTEXT:", worst["expected_context"])
print("RETRIEVED CONTEXT:", worst["retrieved_context"])
print("EXPECTED ANSWER:", worst["expected_answer"])
print("MODEL ANSWER:", worst["answer"])
print("HALLUCINATION RISK:", round(worst["hallucination_risk"], 3))


## 8. Checklist ghi nhớ nhanh

Khi xây LLM/RAG app, hãy tự hỏi:

- [ ] Có trace từng request chưa?
- [ ] Có log prompt, output, token, latency, cost chưa?
- [ ] Với RAG, có log retrieved context chưa?
- [ ] Có evaluation dataset cố định để regression test chưa?
- [ ] Có metrics cho relevance, factuality/hallucination, context quality chưa?
- [ ] Có dashboard hoặc trace viewer để debug từng lỗi chưa?
- [ ] Có quy trình thêm production failure vào eval dataset chưa?

**Câu nhớ nhanh:** Evaluation cho biết app tốt hay dở; observability cho biết vì sao.
